# How the `HolePatternSeal` Code Works

This page describes the model behind `ross.seals.holepattern_seal.HolePatternSeal` and how
the code implements it: the bulk-flow equations, the two gas closures, the solution of the
base flow and of its first-order perturbation, and the convention of the stored
coefficients. [tutorial_seals](../user_guide/tutorial_seals.ipynb) shows how to use the seal;
this page is for readers who want to know what a result means and for developers who change
`holepattern_seal.py` or the gas objects of `gas_model.py`.

The model is a one-control-volume bulk-flow model of the annular clearance in the line of
Nelson {cite}`nelson1984analysis,nelson1985rotordynamic`, with the cells acting as gas
storage as in Kleynhans and Childs {cite}`kleynhans1996acoustic,kleynhans1996`. Section 7
compares it with the hole-pattern measurements of Childs and Wade
{cite}`childs2004rotordynamic`.

**Notation.** $z$ axial coordinate, $\theta$ circumferential angle, $H$ radial clearance,
$R$ rotor radius, $\omega$ rotor speed, $\Omega$ whirl frequency, $u$ and $w$ axial and
circumferential bulk velocities, $G = \rho u$ mass flux, $H_d$ cell depth, $\xi_i$ and
$\xi_e$ entrance and exit loss coefficients. Subscript 0 is the base flow, subscript 1 the
first-order perturbation.

## 1. Model and assumptions

- **Control volume.** The annular clearance, with constant clearance $H$ and flow area
  $2 \pi R H$; the leakage is $G \cdot 2 \pi R H$.
- **Cells.** The cells take no part in the steady flow. Under a perturbation they store gas:
  a cell of depth $H_d$ holds $H_d \rho_1 = H_d \rho_0 \kappa_s p_1$ per unit area, with the
  isentropic compressibility $\kappa_s = 1 / (\rho a^2)$. They enter the model only through
  `cell_depth`. For a hole pattern, use the volume-equivalent depth: hole depth times
  hole-area ratio. `cell_length` and `cell_width` are stored and not used.
- **Friction.** Moody's law for the Fanning factor with the hydraulic diameter $2H$, and the
  shear stress $\tau = \frac{1}{2} \rho f U \mathbf{U}$ that the gas exerts on each wall,
  with $\mathbf{U}$ the velocity of the gas relative to that wall:

  $$
  f_r = 0.001375 \left[1 + \left(10^4 \frac{e_r}{H} + \frac{5 \cdot 10^5 \mu}{\rho H U_r}\right)^{1/3}\right],
  \qquad
  f_s = 0.001375 \left[1 + \left(\frac{10^6 \mu}{\rho H U_s}\right)^{1/3}\right],
  $$

  $$
  \tau_x = \tfrac{1}{2} \rho u \left(\tfrac{1}{4} U_s f_s + U_r f_r\right),
  \qquad
  \tau_\theta = \tfrac{1}{2} \rho \left(\tfrac{1}{4} w U_s f_s + (w - R\omega) U_r f_r\right),
  $$

  with $U_s = \sqrt{u^2 + w^2}$ and $U_r = \sqrt{u^2 + (w - R\omega)^2}$.
  `relative_roughness` is $e_r / H$, the roughness of the rotor divided by the clearance.
  The stator is treated as a smooth wall at a quarter of the Moody value: there is no
  hole-pattern friction law (section 9).
- **Energy.** Adiabatic walls; the rotor does the work $-R \omega \tau_{r\theta}$ on the gas,
  where $\tau_{r\theta} = \frac{1}{2} \rho (w - R\omega) U_r f_r$ is the rotor part of
  $\tau_\theta$; it is negative, and the work positive, while the gas turns slower than the
  rotor.
- **Entrance.** Isentropic acceleration from $(p_{in}, T_{in})$ to the entrance pressure
  $p_2$, then a loss $\xi_i \rho_2 u_2^2 / 2$ of total pressure. The inlet swirl is
  $w_3 = \text{preswirl} \cdot R \omega \, u_3 / u_2$: the flow angle of the accelerated
  stream is kept.
- **Exit.** The last station is brought to rest isentropically with its axial kinetic energy
  (the swirl energy is not recovered), the total pressure loses $\xi_e \rho_4 u_4^2 / 2$, and
  the static pressure $p_5$ that carries the mass flux at that total pressure is matched to
  `outlet_pressure`. $\xi_e$ is a loss, not a recovery.
- **Scope.** Single-phase gas, constant clearance, and quasi-steady entrance and exit: the
  gas upstream and downstream of the seal has no inertia.

## 2. Gas closures

Both closures are objects of `ross/seals/gas_model.py` with the interface of `StateGas`:

| Method | Returns |
|---|---|
| `point(p, T)` | a `GasPoint`: $\rho$, $(\partial\rho/\partial p)_T$, $(\partial\rho/\partial T)_p$, $h$, $c_p$, $(\partial h/\partial p)_T$, $\mu$, $(\partial\mu/\partial p)_T$, $(\partial\mu/\partial T)_p$ |
| `entropy(p, T)` | specific entropy (only differences are used) |
| `temperature_ph(p, h, guess)`, `temperature_ps(p, s, guess)` | the inversions |
| `widened(error)` | a gas object with a wider domain, or the error again |

The solver uses nothing else, so the equations of sections 3 and 4 are the same for both
closures. A state outside the domain of a gas object raises `GasRangeError` (a
`ValueError`) with the pressure, the temperature and the side (`"below"` or `"above"`).
`GasPoint` derives the quantities the equations use:

| Quantity | Expression | Perfect gas |
|---|---|---|
| `rho_p_s` | $(\partial\rho/\partial p)_s = a^{-2} = \rho_p - T \rho_T^2 / (\rho^2 c_p)$; a `ValueError` naming the state where it, $c_p$ or $\rho_p$ is not positive | $1 / (\gamma R T)$ |
| speed of sound `sound_speed` | $a^{-2} = \rho_p - T \rho_T^2 / (\rho^2 c_p)$ | $\sqrt{\gamma R T}$ |
| isentropic compressibility `kappa_s` | $1 / (\rho a^2)$ | $1 / (\gamma p)$ |
| `p_rho`, `p_t` | $1 / \rho_p$, $-\rho_T / \rho_p$ | $R T$, $\rho R$ |
| `h_rho`, `h_t` | $h_p / \rho_p$, $c_p - h_p \rho_T / \rho_p$ | $0$, $c_p$ |
| `mu_rho`, `mu_t` | $\mu_p / \rho_p$, $\mu_T - \mu_p \rho_T / \rho_p$ | $0$, $d\mu/dT$ |

### 2.1 `gas_model="ideal"`: `SutherlandGas`

A perfect gas, $p = \rho R T$ with $R = 8314 / M$ and constant $\gamma$, and the Sutherland
viscosity $\mu = b T^{3/2} / (S + T)$, all in closed form. With `gas_composition`, $M$ comes
from the composition, $\gamma = c_p / c_v$ of the real inlet state, and $b$ and $S$ are
fitted to the equation-of-state viscosity at the inlet pressure between 260 and 380 K.
The fit holds at one pressure and in a fixed temperature range. For a dense gas it can give
$S < 0$; the law then has a pole at $T = -S$, and a state at or below it is a
`GasRangeError`, which fails that speed with a warning.

### 2.2 `gas_model="real"`: `TabulatedGas`

$\rho$, $h$, $s$ and $\mu$ come from the equation of state of `ccp` (`ccp.config.EOS`:
CoolProp's HEOS {cite}`bell2014pure` or REFPROP {cite}`lemmon2018refprop`), tabulated by
`TabulatedGas.from_composition` over the pressure and temperature range of the seal on a
41 x 41 grid. Bicubic interpolating splines give the values and the derivatives, and the
speed of sound follows from the derivatives. The table is never extrapolated.

- **A grid sheared along the inlet isentrope.** Node $(i, j)$ lies at
  $(p_i, T_{lo}(p_i) + \Delta T_j)$, where the cold edge $T_{lo}(p)$ is the inlet isentrope
  5 K colder. The pressures run from 0.9 `outlet_pressure` (or the floor, below) to
  1.001 `inlet_pressure`. The band $\Delta T$ is the same at every pressure; at the lowest
  pressure its top lies 25 K above the warmer of the inlet temperature and the isenthalpic
  temperature, which covers the friction-heated slow trials of the outlet search. A cold edge
  at constant temperature would lie far below the isentrope at the high pressures and can
  cross the saturation line of a pure or nearly pure fluid between nodes that all pass a
  single-phase check; the sheared edge stays next to the states the expansion visits.
- **Imposed phase.** A mixture table is filled through a `ccp.State` with the imposed phase
  `"supercritical_gas"`, which is much faster than a free flash for mixtures and does not
  store the spurious two-phase splits a free flash can return. An imposed phase returns a
  metastable root for a liquid or two-phase state, so the checks below use free flashes. A
  pure-fluid table is filled by free flash, which costs the same there: above the critical
  pressure and below the critical temperature, REFPROP's imposed root is the vapour-like one,
  not the stable one.
- **Single-phase test.** A pure fluid is single phase when $p \ge p_{crit}$ or
  $T > T_{sat}(p)$; this is exact. Below the triple-point pressure, where the gas borders the
  solid, it is single phase when $T > T_{triple}$; the sublimation temperature is lower, so
  this is conservative (CO2 below 0.518 MPa needs more than 216.6 K). A mixture is single
  phase when the imposed-root density equals the free-flash density to $10^{-6}$, or when the
  free flash is a spurious split
  (quality in $[0, 1]$ with saturated liquid and vapour densities within 1 %, ccp's
  criterion) and the two densities agree to 1 %.
- **Inlet check.** An inlet that is not a single-phase gas is refused with a `ValueError`
  before anything else is computed.
- **Floor.** If the isentrope, 5 K colder, is not single phase at 0.9 `outlet_pressure`, the
  lowest pressure at which it is (12 bisections) becomes the lowest pressure of the table,
  and `floor_raised` is True.
- **Guard.** Every pressure node of the cold edge must be single phase; otherwise the table is
  refused with a `ValueError`.
- **Widening.** A flow that leaves the table on the hot side or above its pressure range
  gets a wider table, for that speed only (up to 5 attempts). A trial of the outlet search that
  leaves it on the cold or low-pressure side lies beyond the solution and is treated like a
  choked trial, except inside the final Brent bracket, where it gets a wider table too. If
  the solution itself needs lower pressures, the table is rebuilt with a lower floor, unless
  the floor was raised: the speed then fails with a warning that the expansion may cross the
  phase boundary.
- **Failed speeds.** A speed that fails gets NaN coefficients and a warning (also when the
  speeds are solved in a process pool). A table of three or more speeds cannot hold NaN for
  its pchip interpolation, so such an element raises a `ValueError` that lists the failed
  speeds and the first error.
- **Cost.** A table of a pure fluid or of air takes well under a second; a multi-component
  natural gas takes seconds with REFPROP and up to about a minute with HEOS. A `HybridSeal`
  rebuilds its hole-pattern stage, and so the table, at every step of its interface-pressure
  search.
- **Limitations.** The floor is conservative: the friction-heated flow is warmer than the
  isentrope at the same pressure, so a seal can be refused although its flow would stay a gas.
  For mixtures the guard samples the cold edge at the pressure nodes only, so a narrow
  two-phase band between nodes, or between the edge and the isentrope near the mixture
  critical point, is not detected. Under REFPROP the free flash of a mixture can report a
  spurious two-phase split whose saturated densities differ by more than ccp's 1 % criterion,
  also far from the critical point; the guard then refuses the seal although HEOS accepts it
  (the AGA-8 Ekofisk gas from 9 to 7 MPa at 324 K is refused at 8.06 MPa, 310.6 K). For a
  pure fluid, a state above the critical pressure counts as single phase also below the
  critical temperature. Next to the critical point the spline table can lose thermodynamic
  stability ($c_p$, $(\partial\rho/\partial p)_T$ or $a^{-2}$ not positive), and a flow that
  reaches such a state fails that speed with a warning that names it. The viscosity model of
  HEOS gives no value at some dense, cold states of natural-gas mixtures; such a table is
  refused with a `ValueError` that names the state (REFPROP builds it).

The figure shows the table of a methane seal from 20 to 1 MPa at 300 K: the inlet isentrope
5 K colder meets the saturation curve near 2.9 MPa, so the floor is raised to that pressure
and the band stays in the gas region.

In [1]:
import plotly.io as pio

import ross  # importing ross sets its own default renderer

pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import ccp
import numpy as np
import plotly.graph_objects as go
from CoolProp.CoolProp import PropsSI

from ross.seals.gas_model import TabulatedGas

# HEOS for every equation-of-state call of this page: the outputs do not depend on REFPROP
ccp.config.EOS = "HEOS"

table = TabulatedGas.from_composition({"methane": 1.0}, 20e6, 300.0, 1e6)
print(
    f"floor raised: {table.floor_raised}, lowest table pressure {table.p_min / 1e6:.4f} MPa"
)

pressure = np.linspace(table.p_min, table.p_max, 200)
lower = np.array([table.lower_temperature(p) for p in pressure])
upper = np.array([table.upper_temperature(p) for p in pressure])
s_in = PropsSI("S", "P", 20e6, "T", 300.0, "HEOS::Methane")
p_isentrope = np.linspace(1e6, 20e6, 200)
t_isentrope = [PropsSI("T", "P", p, "S", s_in, "HEOS::Methane") for p in p_isentrope]
p_crit = PropsSI("Pcrit", "HEOS::Methane")
p_saturation = np.linspace(1e6, 0.999 * p_crit, 100)
t_saturation = [PropsSI("T", "P", p, "Q", 1, "HEOS::Methane") for p in p_saturation]

fig = go.Figure()
fig.add_scatter(
    x=lower, y=pressure / 1e6, mode="lines", line=dict(width=1), name="cold edge"
)
fig.add_scatter(
    x=upper,
    y=pressure / 1e6,
    mode="lines",
    fill="tonextx",
    line=dict(width=1),
    name="table band",
)
fig.add_scatter(
    x=t_isentrope, y=p_isentrope / 1e6, mode="lines", name="inlet isentrope"
)
fig.add_scatter(
    x=t_saturation, y=p_saturation / 1e6, mode="lines", name="saturation curve"
)
fig.add_hline(y=table.p_min / 1e6, line_dash="dot", annotation_text="floor")
fig.update_layout(xaxis_title="T (K)", yaxis_title="p (MPa)", height=450)
fig.show()

floor raised: True, lowest table pressure 2.9191 MPa


## 3. Zeroth order: the base flow

The unknowns along $z$ are $p$, $T$ and $w$; the axial velocity is $u = G / \rho(p, T)$, so
mass is conserved exactly. With $u' = -(u / \rho)(\rho_p p' + \rho_T T')$, the momentum and
energy balances of the clearance are

$$
\begin{aligned}
w' &= -\frac{\tau_\theta}{H G}, \\
(1 - u^2 \rho_p)\, p' - u^2 \rho_T\, T' &= -\frac{\tau_x}{H}, \\
\left(h_p - \frac{u^2 \rho_p}{\rho}\right) p' + \left(c_p - \frac{u^2 \rho_T}{\rho}\right) T'
&= -\frac{R \omega \tau_{r\theta}}{H G} - w w',
\end{aligned}
$$

so the total enthalpy $h + (u^2 + w^2)/2$ grows by the work of the rotor. The $2 \times 2$
system for $p'$ and $T'$ is singular at $u = a$. It is integrated by the classical fourth-order
Runge-Kutta method over `nz` equal intervals, starting from station 0.

**Entrance and exit.** Both are the same construction: on an isentrope of total enthalpy
$h_0$, find the subsonic state with $\rho \sqrt{2 (h_0 - h)} = G$. Newton's method uses the
slope $d(\rho u)/dp|_s = (M^2 - 1)/u$; when it does not converge, the sonic pressure of the
isentrope is located, a flux above the sonic flux has no subsonic state, and a flux below it
is bracketed between the sonic and the stagnation pressure and solved by Brent's method to
$10^{-12} G$. At the entrance, the trial pressure $p_2$ on the inlet isentrope fixes $G$ and
station 0 follows at the reduced total pressure. At the exit, the stagnation state of the
last station (total enthalpy $h_4 + u_4^2/2$) gives $p_5$ after the exit loss.

**Outlet-pressure search.** The first trial is $p_2 = (1 - \text{first\_step\_size})\,
p_{in}$, and that step is divided by 10 while the trial already passes the outlet pressure or
chokes. Once the root is bracketed, Brent's method converges $p_5 = p_{out}$ to rounding;
`tolerance` is the largest residual accepted at the end, as a fraction of
$p_{in} - p_{out}$, and `max_iterations` caps the number of trial integrations.

**Choke as a limit.** A trial is choked when $u^2/a^2 > 0.98$ at any Runge-Kutta stage, or when
it has no subsonic entrance or exit state. When the trials choke before $p_5$ reaches the
outlet pressure, bisection finds the smallest unchoked $p_2$ (the largest unchoked mass flux) to $4 \cdot 10^{-16} p_{in}$. That
state is the result, flagged in `seal.choked`: its $p_5$ lies above `outlet_pressure`, and the
leakage and the coefficients are the same for every lower outlet pressure. With $\xi_e > 0$ the
choke is in the exit relation and the last station stays below Mach 1.

The cell below builds the 0.20 mm seal of Childs and Wade {cite}`childs2004rotordynamic`
(Fig. 4 conditions, air as a perfect gas) and solves its base flow with
`solver.base_state`, the diagnostic hook of the solver.

In [3]:
from plotly.subplots import make_subplots

from ross.seals.holepattern_seal import HolePatternSeal
from ross.units import Q_

AIR = {
    "molar_mass": 28.97,
    "gamma": 1.4,
    "sutherland_b": 1.458e-6,
    "sutherland_s": 110.4,
}
CHILDS_WADE = {
    "n": 0,
    "shaft_diameter": 114.721e-3,
    "axial_length": 85.7e-3,
    "relative_roughness": 0.0,
    "cell_length": 3.18e-3,
    "cell_width": 3.18e-3,
    "cell_depth": 3.30e-3 * 0.684,  # hole depth x hole-area ratio
    "inlet_pressure": 70e5,
    "speed": Q_([20200], "RPM"),
    **AIR,
}
FIG4 = dict(
    radial_clearance=0.5 * (0.2115 + 0.2102) * 1e-3,
    preswirl=0.598,
    outlet_pressure=0.49 * 70e5,
    inlet_temperature=292.21,
)
FIG6 = dict(
    radial_clearance=0.5 * (0.1099 + 0.1086) * 1e-3,
    preswirl=0.303,
    outlet_pressure=0.47 * 70e5,
    inlet_temperature=285.02,
)

seal = HolePatternSeal(**CHILDS_WADE, **FIG4)
speed = Q_(20200, "RPM").to("rad/s").m
base = seal.solver.base_state(speed)

print(f"leakage        : {base.leakage:.4f} kg/s")
print(f"choked         : {base.choked}")
print(f"p2, p5         : {base.p2 / 1e5:.3f} bar, {base.p5 / 1e5:.3f} bar")
print(f"trials         : {base.iterations}")

swirl_ratio = base.w / (0.5 * seal.shaft_diameter * speed)
fig = make_subplots(
    rows=1, cols=3, subplot_titles=("p (bar)", "T (K)", "Mach number and swirl ratio")
)
fig.add_scatter(x=1e3 * base.z, y=base.p / 1e5, name="p", row=1, col=1)
fig.add_scatter(x=1e3 * base.z, y=base.T, name="T", row=1, col=2)
fig.add_scatter(x=1e3 * base.z, y=np.sqrt(base.mach2), name="u / a", row=1, col=3)
fig.add_scatter(x=1e3 * base.z, y=swirl_ratio, name="w / (R ω)", row=1, col=3)
fig.update_xaxes(title_text="z (mm)")
fig.update_layout(height=380)
fig.show()

leakage        : 0.7590 kg/s
choked         : False
p2, p5         : 63.347 bar, 34.300 bar
trials         : 19


## 4. First order: the perturbation

A rotor displacement changes the clearance by $h_1 = -e_x \cos\theta - e_y \sin\theta$. Each
of the four components (cosine and sine in $\theta$ and in time) is marched along the seal
with the unknowns $X = (\rho_1, T_1, u_1, w_1)$ at every station, and

$$
p_1 = p_\rho \rho_1 + p_T T_1, \qquad
E_1 = h_\rho \rho_1 + h_T T_1 + u_0 u_1 + w_0 w_1, \qquad
\mu_1 = \mu_\rho \rho_1 + \mu_T T_1 .
$$

With $D_t = \partial_t + (w_0 / R)\, \partial_\theta$, the four equations are

$$
\begin{aligned}
&\partial_t \left[\rho_0 \left(h_1 + H_d \kappa_s p_1\right) + H \rho_1\right]
 + \frac{1}{R} \partial_\theta \left(w_0 H \rho_1 + \rho_0 H w_1 + \rho_0 w_0 h_1\right)
 + \partial_z \left(H u_0 \rho_1 + H \rho_0 u_1 + G h_1\right) = 0, \\
&\rho_0 H D_t E_1 + \rho_0 u_0 H\, \partial_z E_1
 + E_0' \left(H u_0 \rho_1 + \rho_0 H u_1 + \rho_0 u_0 h_1\right)
 - H \partial_t p_1 + R \omega \tau_{r\theta,1} = 0, \\
&\rho_0 H \left(D_t u_1 + u_0 u_1'\right) + H u_0 u_0' \rho_1 + H \rho_0 u_0' u_1
 + \rho_0 u_0 u_0' h_1 + H p_1' + p_0' h_1 + \tau_{x,1} = 0, \\
&\rho_0 H \left(D_t w_1 + u_0 w_1'\right) + H u_0 w_0' \rho_1 + H \rho_0 w_0' u_1
 + \rho_0 u_0 w_0' h_1 + \frac{H}{R} \partial_\theta p_1 + \tau_{\theta,1} = 0,
\end{aligned}
$$

continuity, energy, axial and circumferential momentum. $E_0 = h_0 + (u_0^2 + w_0^2)/2$ is
the total enthalpy of the base flow. In $\partial_z E_1$ and $p_1'$ the derivatives of the
station coefficients ($h_\rho$, $p_\rho$, ...) are station differences. The shear
perturbations $\tau_{k,1}$ are central differences of the shear function of section 1 with
respect to $H$, $\rho$, $\mu$, $u$ and $w$, computed once per station.

Assumptions stated by these equations:

- the cell gas is compressed isentropically ($\kappa_s$), as the adiabatic main flow;
- the gas exchanged with the cells carries the local total enthalpy, so the cells add no term
  to the energy equation;
- the energy equation is the full linearisation of the base-flow one: unsteady and
  circumferential transport of $E_1$, the pressure rate, the perturbed mass flux times
  $E_0'$ and the perturbed rotor work.

The march uses Heun's method over the same `nz` intervals, each stage with the station tables
of its own station.

**Two-point boundary relations.** The entrance state is not zero: it moves along the
mass-flux family of the entrance model, $X(0) \propto (d\rho_3/du_3, dT_3/du_3, 1,
dw_3/du_3)$, one amplitude per component. At the exit the linearised exit relation keeps
$p_5$ fixed, and a choked flow keeps its choke condition. When the land chokes, the axial Mach
number $u/a$ of the last station stays constant {cite}`kleynhans1996acoustic`. When the exit
relation chokes ($\xi_e > 0$), the exit stays sonic: $\rho_4 u_4 / G^*$ stays 1, where $G^*$
is the sonic flux of the isentrope behind the exit loss at the total enthalpy
$h_4 + u_4^2/2$. For a perfect gas the two conditions coincide; for a real gas only the second
is the linearisation of the choke, which the static test of the clearance derivative of the
base flow confirms. The derivatives are central differences. Close to an exit choke a
perturbed state of the difference can have no subsonic exit, and the sonic-exit row, which the
row of $p_5$ tends to there, is used instead. One forced march (the clearance perturbation from a zero
entrance state) and four free marches (unit entrance amplitudes) are superposed, and the four
amplitudes follow from the four exit conditions. The relations are quasi-steady.

**Forces.** $f_x$ and $f_y$ integrate $-p_1$ and the rotor shear perturbation over the rotor
surface (trapezoid over all stations), giving the four components `fx_sin`, `fx_cos`,
`fy_sin` and `fy_cos` at each whirl frequency. All whirl frequencies of one speed are
marched in one vectorised batch on the same base flow.

## 5. Coefficients

At each table point (rotor speed $\omega$, whirl frequency $\Omega$) the element stores the
stiffness and damping at $\Omega$:

$$
k_{xx} = k_{yy} = -f_{x,\cos}(\Omega), \quad k_{xy} = -k_{yx} = f_{y,\cos}(\Omega), \quad
c_{xx} = c_{yy} = \frac{f_{x,\sin}(\Omega)}{\Omega}, \quad
c_{xy} = -c_{yx} = -\frac{f_{y,\sin}(\Omega)}{\Omega},
$$

and zero mass terms, `mxx = myy = mxy = myx = 0`.

- **Why no mass terms.** The cell storage makes the stiffness strongly frequency dependent.
  Written as $K(0) - M \Omega^2$, the fitted $M = (K(0) - K(\Omega))/\Omega^2$ is large and
  negative (cell below). In a modal analysis the rotor multiplies $M$ by $\lambda^2$, not by
  $\Omega^2$, which adds a spurious term and can make the mass matrix indefinite. Storing
  $K(\Omega)$ and $C(\Omega)$ is exact at every tabulated point and is the convention of
  `LabyrinthSeal`, so a `HybridSeal` sums two stages of the same kind.
- **1-D and 2-D tables.** Without `frequency`, the coefficients at each speed are those at
  $\Omega$ = `excitation_ratio * speed`, constant in the frequency of the analysis. An analysis
  that needs the whirl dependence (`run_freq_response`, `run_modal(..., frequency=...)`,
  `matched_whirl=True`) needs a table built with `frequency=`.
- **Zero whirl frequency.** For $|\Omega| < 10^{-9} \max(|\omega|, 1)$ the coefficients are
  evaluated at that small frequency: the damping is its zero-frequency limit, and the
  stiffness equals the static one to rounding.
- **Comparison with the literature.** $K(\Omega)$ of ISOTSEAL-type predictions and of
  measurements {cite}`childs2004rotordynamic` compares directly with `kxx`.

In [4]:
solver = seal.solver
static, synchronous = solver.solve(speed, 0.0), solver.solve(speed, speed)

print(f"K(0)  = {static['kxx']:.4e} N/m,   C(0)  = {static['cxx']:.4e} N s/m")
print(f"K(ω)  = {synchronous['kxx']:.4e} N/m,   C(ω)  = {synchronous['cxx']:.4e} N s/m")
print(f"(K(0) - K(ω)) / ω² = {(static['kxx'] - synchronous['kxx']) / speed**2:.2f} kg")
print(f"stored mxx = {synchronous['mxx']}")

K(0)  = -3.6457e+06 N/m,   C(0)  = 4.3425e+04 N s/m
K(ω)  = 4.3120e+07 N/m,   C(ω)  = 9.3256e+03 N s/m
(K(0) - K(ω)) / ω² = -10.45 kg
stored mxx = 0.0


## 6. Code map

| Object | Role |
|---|---|
| `HolePatternSeal` | the element: validates the inputs, builds the gas object (`seal.gas`) and the solver, solves the speeds (in a process pool above 4 speeds) and stores the coefficients, `seal_leakage`, `p` and `choked` |
| `HolePatternSolver` (`seal.solver`) | plain data and the public entry points: `solve(speed, frequency)`, `solve_row(speed, frequencies)`, `solve_grid(speeds, frequencies)`, `base_state(speed)` and `shear(...)`; retries a speed with a widened table (`_solve_base`), and turns a failure into NaN with a warning |
| `_HolePatternFlow` | the equations of one solve: `base_rhs`, `_integrate` (RK4), `_isentropic_flux_pressure`, `entrance`, `exit_pressure`, `exit_flux_ratio` (the sonic-exit condition), `_trial` and `base_state` (outlet search and choke), `_assemble`; `_station_tables`, `_shear_derivatives`, `_march` (Heun), `_boundary_relations`, `forces` and `coefficients` |
| `HolePatternBaseState` | the base flow at the stations: `p`, `T`, `rho`, `u`, `w`, `mach2`, `mass_flux`, `leakage`, `choked`, `choke_location`, `p2`, `p5`, `iterations`, `gas` |
| `SutherlandGas`, `TabulatedGas` (`gas_model.py`) | the "ideal" and "real" closures (section 2); `TabulatedGas.from_function` builds a table of an analytic gas for tests |
| `GasPoint`, `GasRangeError` | the state at one $(p, T)$ with its derivatives, and the error raised outside a domain |

`seal.p[k]` holds `nz + 4` pressures, $[p_{in}, p_2, p_0 \ldots p_{nz}, p_5]$, which
`plot_pressure_distribution` draws.

## 7. Comparison with Childs and Wade (2004)

Childs and Wade {cite}`childs2004rotordynamic` measured the leakage and the frequency-dependent
coefficients of hole-pattern seals with air at 70 bar. The cell below solves the 0.20 mm seal
(their Fig. 4 conditions) and the 0.10 mm seal (Fig. 6) at 20 200 rpm on a 10 to 300 Hz grid.
The markers are values read from the figures (the paper prints no coefficient tables): the
means of groups of markers read from 300 dpi page images, with the axes calibrated on the
ticks.

In [5]:
cases = {
    "Fig. 4, 0.20 mm": {
        "inputs": FIG4,
        "leakage": 0.409,
        "kxx": [
            (32.0, 6.20),
            (89.3, 12.33),
            (145.7, 19.19),
            (210.4, 27.13),
            (286.1, 36.18),
        ],
        "kxy": [
            (27.9, 21.17),
            (92.9, 16.89),
            (145.0, 13.47),
            (209.7, 10.05),
            (284.0, 8.10),
        ],
        "cxx": [
            (28.6, 32.80),
            (91.8, 27.31),
            (145.0, 21.45),
            (210.8, 16.84),
            (285.4, 12.17),
        ],
    },
    "Fig. 6, 0.10 mm": {
        "inputs": FIG6,
        "leakage": 0.175,
        "kxx": [
            (31.9, 13.55),
            (89.5, 24.82),
            (145.0, 32.93),
            (211.4, 39.38),
            (275.8, 43.97),
        ],
        "cxx": [(144.1, 18.64), (210.9, 11.36), (274.6, 6.94)],
    },
}
scale = {"kxx": 1e6, "kxy": 1e6, "cxx": 1e3}
titles = ("K (MN/m)", "k (MN/m)", "C (kN s/m)")
frequency_hz = np.arange(10.0, 301.0, 10.0)

fig = make_subplots(rows=1, cols=3, subplot_titles=titles)
colors = ("#1f77b4", "#d62728")
for (label, case), color in zip(cases.items(), colors, strict=True):
    grid = HolePatternSeal(
        **CHILDS_WADE, **case["inputs"], frequency=2 * np.pi * frequency_hz
    )
    model = {name: np.array(getattr(grid, name))[0] for name in scale}
    print(
        f"{label}: leakage {grid.seal_leakage[0]:.3f} kg/s, Table 5 {case['leakage']} kg/s, "
        f"ratio {grid.seal_leakage[0] / case['leakage']:.2f}"
    )
    for col, name in enumerate(scale, start=1):
        fig.add_scatter(
            x=frequency_hz,
            y=model[name] / scale[name],
            mode="lines",
            line_color=color,
            name=label,
            legendgroup=label,
            showlegend=col == 1,
            row=1,
            col=col,
        )
        if name in case:
            read_hz, read = np.array(case[name]).T
            found = np.interp(read_hz, frequency_hz, model[name]) / scale[name]
            ratio = found / read
            print(
                f"    {name}: model / measured {ratio.min():.2f} to {ratio.max():.2f}"
            )
            fig.add_scatter(
                x=read_hz,
                y=read,
                mode="markers",
                marker_color=color,
                name=f"{label}, measured",
                legendgroup=label,
                showlegend=col == 1,
                row=1,
                col=col,
            )
fig.update_xaxes(title_text="whirl frequency (Hz)")
fig.update_layout(height=420)
fig.show()

Fig. 4, 0.20 mm: leakage 0.759 kg/s, Table 5 0.409 kg/s, ratio 1.86
    kxx: model / measured -0.41 to 1.13
    kxy: model / measured 1.93 to 2.75
    cxx: model / measured 1.11 to 1.54


Fig. 6, 0.10 mm: leakage 0.308 kg/s, Table 5 0.175 kg/s, ratio 1.76
    kxx: model / measured -0.01 to 1.14
    cxx: model / measured 1.33 to 1.70


The model reproduces the measured frequency trends: $K$ rises and $C$ and $k$ fall with the
whirl frequency, and the effective damping $C - k/\Omega$ changes sign inside the band. $C$
is 11 to 70 % high, $k$ about 2 to 3 times the measured value, and the leakage 1.76 to 1.86
times Table 5; $K$ lies below the measurements at the low frequencies and reaches them at the
top of the band. The stator friction is the main suspect: a quarter of a smooth wall is far
below the friction of a hole-pattern stator (section 9). The "real" closure has no validation here: no dense-gas hole-pattern
measurements were available.

For a dense gas the two closures differ. The cell below compares them for methane from 20 to
10 MPa at 300 K (0.20 mm seal geometry, preswirl 0.2, 15 200 rpm).

In [6]:
methane = dict(
    CHILDS_WADE,
    radial_clearance=0.5 * (0.2115 + 0.2102) * 1e-3,
    inlet_pressure=20e6,
    outlet_pressure=10e6,
    inlet_temperature=300.0,
    preswirl=0.2,
    speed=Q_([15200], "RPM"),
    gas_composition={"methane": 1.0},
)
for name in AIR:
    methane.pop(name)

closures = {
    model: HolePatternSeal(**methane, gas_model=model) for model in ("ideal", "real")
}
print(
    f"{'':>6} | {'leakage (kg/s)':>14} | {'kxx (N/m)':>11} | {'kxy (N/m)':>11} | {'cxx (N s/m)':>11}"
)
for model, item in closures.items():
    print(
        f"{model:>6} | {item.seal_leakage[0]:14.4f} | {item.kxx[0]:11.4e} | "
        f"{item.kxy[0]:11.4e} | {item.cxx[0]:11.4e}"
    )

       | leakage (kg/s) |   kxx (N/m) |   kxy (N/m) | cxx (N s/m)
 ideal |         1.6835 |  4.8615e+07 |  2.5565e+07 |  6.1291e+04
  real |         1.9383 |  6.2617e+07 |  2.7479e+07 |  6.7687e+04


## 8. Changes from earlier ROSS versions

Earlier versions of `HolePatternSeal` had the defects listed in
[petrobras/ross#1475](https://github.com/petrobras/ross/issues/1475). The present formulation
fixes them as follows.

| # | Fixed |
|---|---|
| D1 | the circumferential pressure gradient carries $H/R$ |
| D2 | exact temperature derivative of the Sutherland viscosity |
| D3 | friction derivatives at each station, with that station's velocities |
| D4 | the rotor shear perturbation of the force holds every term |
| D5 | exact viscosity derivative of the stator shear |
| D6 | the base flow conserves total enthalpy with swirl |
| D7 | entrance and exit relations converged, with the sonic limit located |
| D8 | choke is a limit: the result no longer depends on the iteration history |
| D9 | the exit relation uses the local total enthalpy |
| D10 | the force integral includes the first station |
| M1 | two-point boundary relations instead of an initial-value march |
| M2 | the cells store gas instead of acting as an effective clearance |
| M3 | linearised energy equation instead of a frozen total enthalpy |

Kept: M4 (the stator friction of section 1) and M5 (the Sutherland fit of section 2.1). The
stored coefficients also changed convention: `kxx` and `kxy` were the zero-frequency
stiffness, with the frequency dependence in mass terms; they are now the stiffness at the
whirl frequency, with zero mass terms (section 5).

## 9. Limitations and open questions

- **Stator friction (M4).** A quarter of a smooth-wall Moody factor makes the leakage and the
  damping too high (section 7). A per-surface Blasius law fitted to hole-pattern stators, such
  as Eq. 8 of Childs and Wade {cite}`childs2004rotordynamic`, is a candidate.
- **Viscosity of the ideal closure (M5).** The Sutherland law is fitted at the inlet pressure
  and between 260 and 380 K, and can have a pole for a dense gas.
- **Cell compressibility.** The cells are compressed isentropically; isothermal cells
  {cite}`kleynhans1996acoustic` give lower stiffness and damping, and nothing here decides
  between the two.
- **Cell energy.** The gas exchanged with the cells is taken to carry the local total
  enthalpy.
- **Exit.** Only the axial kinetic energy is recovered before the exit loss.
- **Choke location.** The loss coefficients decide whether the flow chokes in the land or in
  the exit relation.
- **Nearly choked exit.** Within about $10^{-3}$ of the choked $p_5$, an unchoked seal uses
  the sonic-exit row of the choked flow instead of the row of $p_5$ (section 4). The
  coefficients then jump at the edge of that band: on the 0.20 mm air seal of the tests, by
  $1.8 \cdot 10^{-3}$ in $k_{xx}$ and $4.7 \cdot 10^{-3}$ in $c_{xy}$.
- **Ideal-gas reference.** $\gamma = c_p / c_v$ of the real inlet state sets the ideal closure,
  so it also sets the size of the differences between the real and the ideal results.
- **Scope.** Single-phase gas and constant clearance only. The real-gas floor is conservative
  (the isentrope, 5 K colder), the mixture guard samples the cold edge at the pressure nodes,
  and section 2.2 lists the states the real-gas table refuses.

## References

```{bibliography}
:filter: docname in docnames
```